# 03 EDA Planning

This notebook locks the strategy for Part 2 before building deeper charts. The goal is not to draw many charts here, but to define the business story: themes, business questions, required joins, metrics, chart ideas, expected insights, recommendations, business value, analysis level, and execution priority.


## 1. Setup

The notebook uses pandas only and exports planning artifacts to `artifacts/eda_planning/` so notebook 04 can use them directly.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 180)

project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "dataset").exists()), Path.cwd())
DATA_DIR = project_root / "dataset"
ARTIFACT_DIR = project_root / "artifacts" / "eda_planning"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Artifact directory:", ARTIFACT_DIR)


Project root: C:\Users\admin\OneDrive - National Economics University\Documents\NCKH\DATATHON\Neu_BRT_Datathon
Artifact directory: C:\Users\admin\OneDrive - National Economics University\Documents\NCKH\DATATHON\Neu_BRT_Datathon\artifacts\eda_planning


## 2. Part 2 Rubric Recap

Part 2 is worth 60 points. The planning target is to push the final EDA story toward prescriptive insights, not just descriptive charts.


In [2]:
rubric_df = pd.DataFrame([
    {
        "criterion": "Visualization quality",
        "what_judges_look_for": "Clear titles, labels, legends, correct chart choice, readable presentation.",
        "planning_implication": "Each chart must answer one business question and use an appropriate visual form.",
    },
    {
        "criterion": "Analytical depth",
        "what_judges_look_for": "Coverage from descriptive to diagnostic, predictive, and prescriptive analysis.",
        "planning_implication": "Each theme should move beyond what happened and explain why, what may happen next, and what to do.",
    },
    {
        "criterion": "Business insight",
        "what_judges_look_for": "Practical findings, specific recommendations, clear link between data and decisions.",
        "planning_implication": "Every selected insight must include an actionable recommendation and business value.",
    },
    {
        "criterion": "Creativity and storytelling",
        "what_judges_look_for": "Non-obvious angles, multi-table reasoning, coherent narrative.",
        "planning_implication": "Prefer connected stories across demand, channels, returns, and operations instead of isolated charts.",
    },
])

display(rubric_df)
display(Markdown("**Planning principle:** prioritize insights that can reach prescriptive level with quantified trade-offs or actions."))


,criterion,what_judges_look_for,planning_implication
0,Visualization quality,"Clear titles, labels, legends, correct chart choice, readable presentation.",Each chart must answer one business question and use an appropriate visual form.
1,Analytical depth,"Coverage from descriptive to diagnostic, predictive, and prescriptive analysis.","Each theme should move beyond what happened and explain why, what may happen next, and what to do."
2,Business insight,"Practical findings, specific recommendations, clear link between data and decisions.",Every selected insight must include an actionable recommendation and business value.
3,Creativity and storytelling,"Non-obvious angles, multi-table reasoning, coherent narrative.","Prefer connected stories across demand, channels, returns, and operations instead of isolated charts."


**Planning principle:** prioritize insights that can reach prescriptive level with quantified trade-offs or actions.

## 3. EDA Tables Available

These are the tables that can support Part 2. Notebook 04 should load and join from this list only.


In [3]:
eda_tables_df = pd.DataFrame([
    {"table": "orders", "role": "Transaction header", "eda_use": "Order timing, status, device, source, customer and geography links."},
    {"table": "order_items", "role": "Transaction line items", "eda_use": "Quantity, unit price, discounts, promotion usage, product-level demand."},
    {"table": "products", "role": "Product dimension", "eda_use": "Category, segment, size, color, price, COGS, margin."},
    {"table": "customers", "role": "Customer dimension", "eda_use": "Age group, gender, acquisition channel, signup date."},
    {"table": "geography", "role": "Geography dimension", "eda_use": "City, district, region analysis."},
    {"table": "returns", "role": "Return events", "eda_use": "Return reasons, return quantities, refund amount, product and order links."},
    {"table": "reviews", "role": "Review events", "eda_use": "Ratings and review title signals after delivery."},
    {"table": "shipments", "role": "Logistics events", "eda_use": "Shipping fee, ship date, delivery date, delivery delay."},
    {"table": "inventory", "role": "Monthly operations snapshot", "eda_use": "Stockout, fill rate, sell-through, overstock, reorder pressure."},
    {"table": "web_traffic", "role": "Daily web behavior", "eda_use": "Sessions, visitors, page views, bounce rate, traffic source."},
    {"table": "promotions", "role": "Promotion dimension", "eda_use": "Promo type, discount value, dates, channel, category applicability."},
    {"table": "sales", "role": "Daily analytical target", "eda_use": "Daily Revenue and COGS time-series trend and seasonality."},
])

display(eda_tables_df)


,table,role,eda_use
0,orders,Transaction header,"Order timing, status, device, source, customer and geography links."
1,order_items,Transaction line items,"Quantity, unit price, discounts, promotion usage, product-level demand."
2,products,Product dimension,"Category, segment, size, color, price, COGS, margin."
3,customers,Customer dimension,"Age group, gender, acquisition channel, signup date."
4,geography,Geography dimension,"City, district, region analysis."
5,returns,Return events,"Return reasons, return quantities, refund amount, product and order links."
6,reviews,Review events,Ratings and review title signals after delivery.
7,shipments,Logistics events,"Shipping fee, ship date, delivery date, delivery delay."
8,inventory,Monthly operations snapshot,"Stockout, fill rate, sell-through, overstock, reorder pressure."
9,web_traffic,Daily web behavior,"Sessions, visitors, page views, bounce rate, traffic source."


## 4. Four Major EDA Themes

The final Part 2 report should be built around four connected themes. Each theme has a clear business question and a path toward action.


In [4]:
themes_df = pd.DataFrame([
    {
        "theme_id": "T1",
        "theme": "Revenue and demand",
        "core_business_question": "Where is revenue growth coming from, and which demand patterns should drive planning?",
        "why_it_matters": "Revenue timing and segment mix guide inventory allocation, campaign timing, and forecasting assumptions.",
    },
    {
        "theme_id": "T2",
        "theme": "Customer and channel",
        "core_business_question": "Which customer and acquisition channels convert into valuable orders, and where is traffic quality weak?",
        "why_it_matters": "Channel mix affects marketing spend, conversion quality, repeat purchase, and customer targeting.",
    },
    {
        "theme_id": "T3",
        "theme": "Returns and customer experience",
        "core_business_question": "Which products, sizes, and delivery experiences create avoidable returns or poor ratings?",
        "why_it_matters": "Returns reduce net revenue and indicate product fit, quality, or logistics problems.",
    },
    {
        "theme_id": "T4",
        "theme": "Inventory and operations",
        "core_business_question": "Where are stockouts, low fill rate, or overstock limiting revenue and service quality?",
        "why_it_matters": "Operational constraints determine whether demand can be fulfilled profitably.",
    },
])

display(themes_df)


,theme_id,theme,core_business_question,why_it_matters
0,T1,Revenue and demand,"Where is revenue growth coming from, and which demand patterns should drive planning?","Revenue timing and segment mix guide inventory allocation, campaign timing, and forecasting assumptions."
1,T2,Customer and channel,"Which customer and acquisition channels convert into valuable orders, and where is traffic quality weak?","Channel mix affects marketing spend, conversion quality, repeat purchase, and customer targeting."
2,T3,Returns and customer experience,"Which products, sizes, and delivery experiences create avoidable returns or poor ratings?","Returns reduce net revenue and indicate product fit, quality, or logistics problems."
3,T4,Inventory and operations,"Where are stockouts, low fill rate, or overstock limiting revenue and service quality?",Operational constraints determine whether demand can be fulfilled profitably.


## 5. Complete EDA Planning Table

Each planned insight must be attached to at least one clear business question, a concrete join plan, metrics, chart type, expected insight, action, and business value. This table is the blueprint for notebook 04.


In [5]:
eda_plan_rows = [
    {
        "insight_id": "I1",
        "priority": 1,
        "theme_id": "T1",
        "theme": "Revenue and demand",
        "business_question": "Which months, categories, and product segments contribute most to revenue and gross margin growth?",
        "tables_used": "orders, order_items, products",
        "join_logic": "order_items.order_id -> orders.order_id; order_items.product_id -> products.product_id; aggregate by order_date month, category, segment.",
        "metrics": "gross_revenue=sum(quantity*unit_price); discount=sum(discount_amount); units=sum(quantity); gross_margin=sum(quantity*(unit_price-cogs)); margin_rate=gross_margin/gross_revenue.",
        "chart_type": "Monthly line chart plus stacked bar by category/segment; optional heatmap for month x category.",
        "expected_insight": "A small set of categories or segments likely explains most revenue and margin movement across time.",
        "expected_actionable_recommendation": "Prioritize stock and campaign budget for high-revenue and high-margin segments before seasonal peaks; reduce emphasis on low-margin volume drivers.",
        "business_value": "Improves revenue planning, assortment focus, and profit-aware campaign allocation.",
        "analysis_level": "descriptive, diagnostic, prescriptive",
    },
    {
        "insight_id": "I2",
        "priority": 2,
        "theme_id": "T1",
        "theme": "Revenue and demand",
        "business_question": "Do daily sales and web traffic show seasonality or leading signals that can support demand planning?",
        "tables_used": "sales, web_traffic",
        "join_logic": "Join sales.Date to web_traffic.date after aggregating web traffic across traffic_source per day.",
        "metrics": "daily_revenue; daily_cogs; gross_margin=Revenue-COGS; sessions; unique_visitors; page_views; bounce_rate; lagged sessions; revenue_per_session.",
        "chart_type": "Time-series line chart with rolling averages; scatter plot of lagged sessions vs revenue; seasonal decomposition-style month/day summary.",
        "expected_insight": "Traffic volume or quality may act as a short-term demand signal before revenue peaks.",
        "expected_actionable_recommendation": "Use traffic spikes and low-bounce sources as early signals for inventory and campaign readiness during high-demand windows.",
        "business_value": "Connects marketing demand signals with forecasting and inventory preparation.",
        "analysis_level": "descriptive, predictive, prescriptive",
    },
    {
        "insight_id": "I3",
        "priority": 3,
        "theme_id": "T2",
        "theme": "Customer and channel",
        "business_question": "Which order sources, devices, and customer acquisition channels create the strongest order value and repeat behavior?",
        "tables_used": "orders, order_items, customers",
        "join_logic": "orders.customer_id -> customers.customer_id; order_items.order_id -> orders.order_id; aggregate by order_source, device_type, acquisition_channel, age_group.",
        "metrics": "orders; unique_customers; revenue=sum(quantity*unit_price); average_order_value; orders_per_customer; repeat_customer_rate; discount_rate.",
        "chart_type": "Segmented bar chart for AOV and repeat rate; bubble chart for revenue vs repeat rate sized by customers.",
        "expected_insight": "Some channels may bring high traffic or many orders but weaker repeat behavior or lower order value.",
        "expected_actionable_recommendation": "Shift acquisition spend toward channels with stronger repeat rate and AOV; tune offers for low-value but high-volume channels.",
        "business_value": "Improves marketing ROI and customer targeting quality.",
        "analysis_level": "descriptive, diagnostic, prescriptive",
    },
    {
        "insight_id": "I4",
        "priority": 4,
        "theme_id": "T2",
        "theme": "Customer and channel",
        "business_question": "Which traffic sources produce high engagement versus high bounce, and how does that relate to revenue opportunities?",
        "tables_used": "web_traffic, orders",
        "join_logic": "Aggregate web_traffic by date and traffic_source; compare to orders by order_date and order_source where channel names can be mapped or analyzed separately by date.",
        "metrics": "sessions; unique_visitors; bounce_rate; avg_session_duration_sec; orders by source; revenue proxy from order_items if joined in notebook 04.",
        "chart_type": "Traffic-source quality bar chart; trend lines for sessions and bounce rate by source.",
        "expected_insight": "Low-bounce sources may be better for conversion quality, while high-volume sources may need landing page or targeting improvement.",
        "expected_actionable_recommendation": "Increase spend or retention campaigns on low-bounce sources; audit landing pages and creative for high-bounce sources.",
        "business_value": "Makes marketing optimization more evidence-based than using sessions alone.",
        "analysis_level": "descriptive, diagnostic, prescriptive",
    },
    {
        "insight_id": "I5",
        "priority": 5,
        "theme_id": "T3",
        "theme": "Returns and customer experience",
        "business_question": "Which categories, sizes, or segments have avoidable return pressure, and what are the main reasons?",
        "tables_used": "returns, order_items, products",
        "join_logic": "returns.product_id -> products.product_id; order_items.product_id -> products.product_id; compare return records to order-item rows by category, segment, size.",
        "metrics": "return_records; return_quantity; refund_amount; order_item_rows; return_rate=return_records/order_item_rows; top return_reason share.",
        "chart_type": "Return-rate bar chart by size/category; stacked bar of return reasons; Pareto chart of refund amount.",
        "expected_insight": "Return risk may concentrate in specific sizes or categories, with reasons such as wrong_size or defective explaining most avoidable loss.",
        "expected_actionable_recommendation": "Improve size guidance and quality checks for high-return size/category combinations; target product pages with clearer fit information.",
        "business_value": "Reduces refund loss, reverse logistics cost, and customer friction.",
        "analysis_level": "descriptive, diagnostic, prescriptive",
    },
    {
        "insight_id": "I6",
        "priority": 6,
        "theme_id": "T3",
        "theme": "Returns and customer experience",
        "business_question": "Does delivery delay or shipping cost relate to lower ratings or higher return risk?",
        "tables_used": "shipments, reviews, returns, orders, products",
        "join_logic": "shipments.order_id -> orders.order_id; reviews.order_id/product_id -> order/product; returns.order_id/product_id -> order/product; add products for category/segment.",
        "metrics": "delivery_delay_days=delivery_date-ship_date; rating; return_flag; refund_amount; shipping_fee; return_rate by delay bucket; average rating by delay bucket.",
        "chart_type": "Box plot or line chart by delay bucket; dual bar chart for return rate and average rating.",
        "expected_insight": "Longer delivery delays may align with lower ratings and higher returns in specific categories or regions.",
        "expected_actionable_recommendation": "Set delay thresholds for proactive customer communication or shipping upgrades on high-value orders.",
        "business_value": "Improves customer experience while quantifying logistics trade-offs.",
        "analysis_level": "diagnostic, predictive, prescriptive",
    },
    {
        "insight_id": "I7",
        "priority": 7,
        "theme_id": "T4",
        "theme": "Inventory and operations",
        "business_question": "Which categories or segments suffer from stockouts or low fill rate during high-demand periods?",
        "tables_used": "inventory, products, order_items, orders",
        "join_logic": "inventory.product_id -> products.product_id; order_items.product_id -> products.product_id; order_items.order_id -> orders.order_id; align inventory month with order_date month.",
        "metrics": "stockout_days; fill_rate; stockout_flag; units_sold; monthly revenue; lost revenue proxy; days_of_supply.",
        "chart_type": "Month x category heatmap for stockout days; scatter plot of fill rate vs revenue; line chart of demand vs stockout days.",
        "expected_insight": "Demand peaks may overlap with low fill rate or stockout days, creating missed revenue opportunities.",
        "expected_actionable_recommendation": "Reorder earlier for high-demand categories with low days_of_supply; set safety-stock thresholds before seasonal peaks.",
        "business_value": "Protects revenue and improves fulfillment reliability.",
        "analysis_level": "descriptive, diagnostic, predictive, prescriptive",
    },
    {
        "insight_id": "I8",
        "priority": 8,
        "theme_id": "T4",
        "theme": "Inventory and operations",
        "business_question": "Where is overstock tying up capital without enough sell-through?",
        "tables_used": "inventory, products",
        "join_logic": "inventory.product_id -> products.product_id; aggregate by category, segment, month.",
        "metrics": "overstock_flag; stock_on_hand; days_of_supply; sell_through_rate; units_received; units_sold.",
        "chart_type": "Quadrant chart: days_of_supply vs sell_through_rate; bar chart of overstock share by segment.",
        "expected_insight": "Some categories may show high stock on hand and low sell-through, indicating markdown or replenishment adjustment opportunities.",
        "expected_actionable_recommendation": "Use targeted markdowns or pause replenishment for overstocked low-sell-through products.",
        "business_value": "Reduces working-capital lockup and markdown waste.",
        "analysis_level": "diagnostic, prescriptive",
    },
]

eda_plan_df = pd.DataFrame(eda_plan_rows)
display(eda_plan_df)


,insight_id,priority,theme_id,theme,business_question,tables_used,join_logic,metrics,chart_type,expected_insight,expected_actionable_recommendation,business_value,analysis_level
0,I1,1,T1,Revenue and demand,"Which months, categories, and product segments contribute most to revenue and gross margin growth?","orders, order_items, products","order_items.order_id -> orders.order_id; order_items.product_id -> products.product_id; aggregate by order_date month, category, segment.",gross_revenue=sum(quantity*unit_price); discount=sum(discount_amount); units=sum(quantity); gross_margin=sum(quantity*(unit_price-cogs)); margin_rate=gross_margin/gross_revenue.,Monthly line chart plus stacked bar by category/segment; optional heatmap for month x category.,A small set of categories or segments likely explains most revenue and margin movement across time.,Prioritize stock and campaign budget for high-revenue and high-margin segments before seasonal peaks; reduce emphasis on low-margin volume drivers.,"Improves revenue planning, assortment focus, and profit-aware campaign allocation.","descriptive, diagnostic, prescriptive"
1,I2,2,T1,Revenue and demand,Do daily sales and web traffic show seasonality or leading signals that can support demand planning?,"sales, web_traffic",Join sales.Date to web_traffic.date after aggregating web traffic across traffic_source per day.,daily_revenue; daily_cogs; gross_margin=Revenue-COGS; sessions; unique_visitors; page_views; bounce_rate; lagged sessions; revenue_per_session.,Time-series line chart with rolling averages; scatter plot of lagged sessions vs revenue; seasonal decomposition-style month/day summary.,Traffic volume or quality may act as a short-term demand signal before revenue peaks.,Use traffic spikes and low-bounce sources as early signals for inventory and campaign readiness during high-demand windows.,Connects marketing demand signals with forecasting and inventory preparation.,"descriptive, predictive, prescriptive"
2,I3,3,T2,Customer and channel,"Which order sources, devices, and customer acquisition channels create the strongest order value and repeat behavior?","orders, order_items, customers","orders.customer_id -> customers.customer_id; order_items.order_id -> orders.order_id; aggregate by order_source, device_type, acquisition_channel, age_group.",orders; unique_customers; revenue=sum(quantity*unit_price); average_order_value; orders_per_customer; repeat_customer_rate; discount_rate.,Segmented bar chart for AOV and repeat rate; bubble chart for revenue vs repeat rate sized by customers.,Some channels may bring high traffic or many orders but weaker repeat behavior or lower order value.,Shift acquisition spend toward channels with stronger repeat rate and AOV; tune offers for low-value but high-volume channels.,Improves marketing ROI and customer targeting quality.,"descriptive, diagnostic, prescriptive"
3,I4,4,T2,Customer and channel,"Which traffic sources produce high engagement versus high bounce, and how does that relate to revenue opportunities?","web_traffic, orders",Aggregate web_traffic by date and traffic_source; compare to orders by order_date and order_source where channel names can be mapped or analyzed separately by date.,sessions; unique_visitors; bounce_rate; avg_session_duration_sec; orders by source; revenue proxy from order_items if joined in notebook 04.,Traffic-source quality bar chart; trend lines for sessions and bounce rate by source.,"Low-bounce sources may be better for conversion quality, while high-volume sources may need landing page or targeting improvement.",Increase spend or retention campaigns on low-bounce sources; audit landing pages and creative for high-bounce sources.,Makes marketing optimization more evidence-based than using sessions alone.,"descriptive, diagnostic, prescriptive"
4,I5,5,T3,Returns and customer experience,"Which categories, sizes, or segments have avoidable return pressure, and what are the main reasons?","returns, order_items, produ

## 6. Planned Charts

This is the chart backlog for notebook 04. It avoids random charting by tying each visual to an insight and business question.


In [6]:
chart_plan_df = eda_plan_df[[
    "insight_id", "priority", "theme", "business_question", "chart_type", "metrics", "tables_used"
]].copy()
chart_plan_df = chart_plan_df.rename(columns={"chart_type": "planned_chart"})

display(chart_plan_df.sort_values("priority"))


,insight_id,priority,theme,business_question,planned_chart,metrics,tables_used
0,I1,1,Revenue and demand,"Which months, categories, and product segments contribute most to revenue and gross margin growth?",Monthly line chart plus stacked bar by category/segment; optional heatmap for month x category.,gross_revenue=sum(quantity*unit_price); discount=sum(discount_amount); units=sum(quantity); gross_margin=sum(quantity*(unit_price-cogs)); margin_rate=gross_margin/gross_revenue.,"orders, order_items, products"
1,I2,2,Revenue and demand,Do daily sales and web traffic show seasonality or leading signals that can support demand planning?,Time-series line chart with rolling averages; scatter plot of lagged sessions vs revenue; seasonal decomposition-style month/day summary.,daily_revenue; daily_cogs; gross_margin=Revenue-COGS; sessions; unique_visitors; page_views; bounce_rate; lagged sessions; revenue_per_session.,"sales, web_traffic"
2,I3,3,Customer and channel,"Which order sources, devices, and customer acquisition channels create the strongest order value and repeat behavior?",Segmented bar chart for AOV and repeat rate; bubble chart for revenue vs repeat rate sized by customers.,orders; unique_customers; revenue=sum(quantity*unit_price); average_order_value; orders_per_customer; repeat_customer_rate; discount_rate.,"orders, order_items, customers"
3,I4,4,Customer and channel,"Which traffic sources produce high engagement versus high bounce, and how does that relate to revenue opportunities?",Traffic-source quality bar chart; trend lines for sessions and bounce rate by source.,sessions; unique_visitors; bounce_rate; avg_session_duration_sec; orders by source; revenue proxy from order_items if joined in notebook 04.,"web_traffic, orders"
4,I5,5,Returns and customer experience,"Which categories, sizes, or segments have avoidable return pressure, and what are the main reasons?",Return-rate bar chart by size/category; stacked bar of return reasons; Pareto chart of refund amount.,return_records; return_quantity; refund_amount; order_item_rows; return_rate=return_records/order_item_rows; top return_reason share.,"returns, order_items, products"
5,I6,6,Returns and customer experience,Does delivery delay or shipping cost relate to lower ratings or higher return risk?,Box plot or line chart by delay bucket; dual bar chart for return rate and average rating.,delivery_delay_days=delivery_date-ship_date; rating; return_flag; refund_amount; shipping_fee; return_rate by delay bucket; average rating by delay bucket.,"shipments, reviews, returns, orders, products"
6,I7,7,Inventory and operations,Which categories or segments suffer from stockouts or low fill rate during high-demand periods?,Month x category heatmap for stockout days; scatter plot of fill rate vs revenue; line chart of demand vs stockout days.,stockout_days; fill_rate; stockout_flag; units_sold; monthly revenue; lost revenue proxy; days_of_supply.,"inventory, products, order_items, orders"
7,I8,8,Inventory and operations,Where is overstock tying up capital without enough sell-through?,Quadrant chart: days_of_supply vs sell_through_rate; bar chart of overstock share by segment.,overstock_flag; stock_on_hand; days_of_supply; sell_through_rate; units_received; units_sold.,"inventory, products"


## 7. Joins To Prepare In Notebook 04

These joins should be implemented once in notebook 04 and reused across charts. Preparing them cleanly will reduce repeated code and reduce the risk of inconsistent metrics.


In [7]:
join_plan_rows = [
    {
        "join_id": "J1_order_lines_enriched",
        "purpose": "Revenue, demand, product mix, channel, and customer segmentation.",
        "base_table": "order_items",
        "joins": "order_items.order_id -> orders.order_id; order_items.product_id -> products.product_id; orders.customer_id -> customers.customer_id; orders.zip -> geography.zip.",
        "output_grain": "One row per order item line.",
        "used_by_insights": "I1, I3, I7",
    },
    {
        "join_id": "J2_daily_sales_traffic",
        "purpose": "Demand trend, seasonality, and web traffic leading-signal analysis.",
        "base_table": "sales",
        "joins": "sales.Date -> aggregated web_traffic.date after grouping traffic across traffic_source per day.",
        "output_grain": "One row per date.",
        "used_by_insights": "I2",
    },
    {
        "join_id": "J3_traffic_source_quality",
        "purpose": "Traffic-source quality and channel comparison.",
        "base_table": "web_traffic",
        "joins": "Optional date-level comparison to orders.order_date and order_source; keep separate if channel naming does not map cleanly.",
        "output_grain": "One row per date and traffic_source.",
        "used_by_insights": "I4",
    },
    {
        "join_id": "J4_returns_enriched",
        "purpose": "Return reason, return rate, refund pressure, and product-fit analysis.",
        "base_table": "returns",
        "joins": "returns.product_id -> products.product_id; returns.order_id -> orders.order_id; compare numerator to order_items joined with products for denominator.",
        "output_grain": "One row per return record plus denominator tables by product/category/size.",
        "used_by_insights": "I5",
    },
    {
        "join_id": "J5_customer_experience",
        "purpose": "Delivery delay, rating, and return-risk analysis.",
        "base_table": "shipments",
        "joins": "shipments.order_id -> orders.order_id; reviews.order_id/product_id -> order/product; returns.order_id/product_id -> order/product; products by product_id.",
        "output_grain": "Order-product level where possible; order level for delay metrics.",
        "used_by_insights": "I6",
    },
    {
        "join_id": "J6_inventory_monthly_enriched",
        "purpose": "Stockout, fill rate, overstock, reorder, and sell-through operations analysis.",
        "base_table": "inventory",
        "joins": "inventory.product_id -> products.product_id; optionally align inventory year/month with order_lines_enriched order month.",
        "output_grain": "One row per product per month.",
        "used_by_insights": "I7, I8",
    },
]

join_plan_df = pd.DataFrame(join_plan_rows)
display(join_plan_df)


,join_id,purpose,base_table,joins,output_grain,used_by_insights
0,J1_order_lines_enriched,"Revenue, demand, product mix, channel, and customer segmentation.",order_items,order_items.order_id -> orders.order_id; order_items.product_id -> products.product_id; orders.customer_id -> customers.customer_id; orders.zip -> geography.zip.,One row per order item line.,"I1, I3, I7"
1,J2_daily_sales_traffic,"Demand trend, seasonality, and web traffic leading-signal analysis.",sales,sales.Date -> aggregated web_traffic.date after grouping traffic across traffic_source per day.,One row per date.,I2
2,J3_traffic_source_quality,Traffic-source quality and channel comparison.,web_traffic,Optional date-level comparison to orders.order_date and order_source; keep separate if channel naming does not map cleanly.,One row per date and traffic_source.,I4
3,J4_returns_enriched,"Return reason, return rate, refund pressure, and product-fit analysis.",returns,returns.product_id -> products.product_id; returns.order_id -> orders.order_id; compare numerator to order_items joined with products for denominator.,One row per return record plus denominator tables by product/category/size.,I5
4,J5_customer_experience,"Delivery delay, rating, and return-risk analysis.",shipments,shipments.order_id -> orders.order_id; reviews.order_id/product_id -> order/product; returns.order_id/product_id -> order/product; products by product_id.,Order-product level where possible; order level for delay metrics.,I6
5,J6_inventory_monthly_enriched,"Stockout, fill rate, overstock, reorder, and sell-through operations analysis.",inventory,inventory.product_id -> products.product_id; optionally align inventory year/month with order_lines_enriched order month.,One row per product per month.,"I7, I8"


## 8. Main Insights For The Report

The report should include 4 to 6 main insights. The first six below are the recommended report candidates because they cover revenue, marketing, returns, customer experience, and operations.


In [8]:
report_insights_df = eda_plan_df[eda_plan_df["insight_id"].isin(["I1", "I2", "I3", "I5", "I6", "I7"])].copy()
report_insights_df = report_insights_df[[
    "insight_id", "priority", "theme", "business_question", "expected_insight",
    "expected_actionable_recommendation", "business_value", "analysis_level"
]].sort_values("priority")

display(report_insights_df)


,insight_id,priority,theme,business_question,expected_insight,expected_actionable_recommendation,business_value,analysis_level
0,I1,1,Revenue and demand,"Which months, categories, and product segments contribute most to revenue and gross margin growth?",A small set of categories or segments likely explains most revenue and margin movement across time.,Prioritize stock and campaign budget for high-revenue and high-margin segments before seasonal peaks; reduce emphasis on low-margin volume drivers.,"Improves revenue planning, assortment focus, and profit-aware campaign allocation.","descriptive, diagnostic, prescriptive"
1,I2,2,Revenue and demand,Do daily sales and web traffic show seasonality or leading signals that can support demand planning?,Traffic volume or quality may act as a short-term demand signal before revenue peaks.,Use traffic spikes and low-bounce sources as early signals for inventory and campaign readiness during high-demand windows.,Connects marketing demand signals with forecasting and inventory preparation.,"descriptive, predictive, prescriptive"
2,I3,3,Customer and channel,"Which order sources, devices, and customer acquisition channels create the strongest order value and repeat behavior?",Some channels may bring high traffic or many orders but weaker repeat behavior or lower order value.,Shift acquisition spend toward channels with stronger repeat rate and AOV; tune offers for low-value but high-volume channels.,Improves marketing ROI and customer targeting quality.,"descriptive, diagnostic, prescriptive"
4,I5,5,Returns and customer experience,"Which categories, sizes, or segments have avoidable return pressure, and what are the main reasons?","Return risk may concentrate in specific sizes or categories, with reasons such as wrong_size or defective explaining most avoidable loss.",Improve size guidance and quality checks for high-return size/category combinations; target product pages with clearer fit information.,"Reduces refund loss, reverse logistics cost, and customer friction.","descriptive, diagnostic, prescriptive"
5,I6,6,Returns and customer experience,Does delivery delay or shipping cost relate to lower ratings or higher return risk?,Longer delivery delays may align with lower ratings and higher returns in specific categories or regions.,Set delay thresholds for proactive customer communication or shipping upgrades on high-value orders.,Improves customer experience while quantifying logistics trade-offs.,"diagnostic, predictive, prescriptive"
6,I7,7,Inventory and operations,Which categories or segments suffer from stockouts or low fill rate during high-demand periods?,"Demand peaks may overlap with low fill rate or stockout days, creating missed revenue opportunities.",Reorder earlier for high-demand categories with low days_of_supply; set safety-stock thresholds before seasonal peaks.,Protects revenue and improves fulfillment reliability.,"descriptive, diagnostic, predictive, prescriptive"


## 9. Analytical Level Coverage

This table checks whether the planned story covers descriptive, diagnostic, predictive, and prescriptive layers. The final report should not stop at describing patterns.


In [9]:
analysis_levels = ["descriptive", "diagnostic", "predictive", "prescriptive"]
coverage_rows = []
for level in analysis_levels:
    subset = eda_plan_df[eda_plan_df["analysis_level"].str.contains(level, case=False, na=False)]
    coverage_rows.append({
        "analysis_level": level,
        "insight_count": len(subset),
        "insight_ids": ", ".join(subset["insight_id"].tolist()),
        "coverage_note": "Covered" if len(subset) > 0 else "Missing",
    })

analysis_coverage_df = pd.DataFrame(coverage_rows)
display(analysis_coverage_df)


,analysis_level,insight_count,insight_ids,coverage_note
0,descriptive,6,"I1, I2, I3, I4, I5, I7",Covered
1,diagnostic,7,"I1, I3, I4, I5, I6, I7, I8",Covered
2,predictive,3,"I2, I6, I7",Covered
3,prescriptive,8,"I1, I2, I3, I4, I5, I6, I7, I8",Covered


## 10. Execution Priority

Notebook 04 should build insights in this order. The first priority is to build the reusable enriched tables, then produce the highest-value report insights.


In [10]:
priority_df = eda_plan_df[[
    "priority", "insight_id", "theme", "business_question", "tables_used", "chart_type", "analysis_level"
]].sort_values("priority").reset_index(drop=True)

display(priority_df)


,priority,insight_id,theme,business_question,tables_used,chart_type,analysis_level
0,1,I1,Revenue and demand,"Which months, categories, and product segments contribute most to revenue and gross margin growth?","orders, order_items, products",Monthly line chart plus stacked bar by category/segment; optional heatmap for month x category.,"descriptive, diagnostic, prescriptive"
1,2,I2,Revenue and demand,Do daily sales and web traffic show seasonality or leading signals that can support demand planning?,"sales, web_traffic",Time-series line chart with rolling averages; scatter plot of lagged sessions vs revenue; seasonal decomposition-style month/day summary.,"descriptive, predictive, prescriptive"
2,3,I3,Customer and channel,"Which order sources, devices, and customer acquisition channels create the strongest order value and repeat behavior?","orders, order_items, customers",Segmented bar chart for AOV and repeat rate; bubble chart for revenue vs repeat rate sized by customers.,"descriptive, diagnostic, prescriptive"
3,4,I4,Customer and channel,"Which traffic sources produce high engagement versus high bounce, and how does that relate to revenue opportunities?","web_traffic, orders",Traffic-source quality bar chart; trend lines for sessions and bounce rate by source.,"descriptive, diagnostic, prescriptive"
4,5,I5,Returns and customer experience,"Which categories, sizes, or segments have avoidable return pressure, and what are the main reasons?","returns, order_items, products",Return-rate bar chart by size/category; stacked bar of return reasons; Pareto chart of refund amount.,"descriptive, diagnostic, prescriptive"
5,6,I6,Returns and customer experience,Does delivery delay or shipping cost relate to lower ratings or higher return risk?,"shipments, reviews, returns, orders, products",Box plot or line chart by delay bucket; dual bar chart for return rate and average rating.,"diagnostic, predictive, prescriptive"
6,7,I7,Inventory and operations,Which categories or segments suffer from stockouts or low fill rate during high-demand periods?,"inventory, products, order_items, orders",Month x category heatmap for stockout days; scatter plot of fill rate vs revenue; line chart of demand vs stockout days.,"descriptive, diagnostic, predictive, prescriptive"
7,8,I8,Inventory and operations,Where is overstock tying up capital without enough sell-through?,"inventory, products",Quadrant chart: days_of_supply vs sell_through_rate; bar chart of overstock share by segment.,"diagnostic, prescriptive"


## 11. Export Planning Outputs

The following CSV files are the required outputs from this planning notebook:

- `eda_plan.csv`: complete EDA planning table
- `chart_plan.csv`: planned chart list
- `join_plan.csv`: joins to prepare in notebook 04
- `report_insights.csv`: recommended 4-6 report insights
- `analysis_coverage.csv`: descriptive/diagnostic/predictive/prescriptive coverage
- `priority_plan.csv`: execution order


In [11]:
exports = {
    "eda_plan.csv": eda_plan_df,
    "chart_plan.csv": chart_plan_df,
    "join_plan.csv": join_plan_df,
    "report_insights.csv": report_insights_df,
    "analysis_coverage.csv": analysis_coverage_df,
    "priority_plan.csv": priority_df,
}

for filename, df in exports.items():
    out_path = ARTIFACT_DIR / filename
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print("Saved", out_path.relative_to(project_root))


Saved artifacts\eda_planning\eda_plan.csv
Saved artifacts\eda_planning\chart_plan.csv
Saved artifacts\eda_planning\join_plan.csv
Saved artifacts\eda_planning\report_insights.csv
Saved artifacts\eda_planning\analysis_coverage.csv
Saved artifacts\eda_planning\priority_plan.csv


## 12. Planning Decision

Notebook 04 should focus on validating and quantifying the prioritized insights instead of exploring every possible chart. The final EDA story should connect demand, customer/channel quality, returns/customer experience, and inventory operations into a prescriptive business narrative.
